# True-plan samples versus an empirical OT reference

These are different experiments, even though both reference couplings are optimal at $u=1/2$.

1. **Paired:** draw $X_i\sim\alpha$ and set $Y_i=T(X_i)$. The observations are iid from the population OT plan. The observed permutation is the identity.
2. **Independent marginals:** draw $X_i\sim\alpha$ and $Z_i\sim\alpha$ independently, set $Y_i=T(Z_i)$, then compute a reference permutation $\tau_*$ by OT at $u=1/2$. This empirical coupling is not an iid sample from the population plan.

For the second experiment subtract $n^{-1}\sum_iX_i^\top A(u)Y_{\tau_*(i)}$. Keep $\tau_*$ fixed while varying $u$. Both finite-sample gaps are convex and piecewise linear. Their curve shapes and finite-sample offsets can differ greatly.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "iot_gap").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import matplotlib.pyplot as plt
from iot_gap import (cosine_map, annulus_map, sample_pairs, sample_independent_marginals,
                     optimal_assignment, gap_curve, sampling_gap_curve)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
US = np.linspace(0, 1, 51)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharex=True)
for ax, delta in zip(axes, [0, .1, .32]):
    Xp, Yp = sample_pairs("cosine", 100, seed=7, delta=delta)
    Xi, Yi = sample_independent_marginals("cosine", 100, seed=7, delta=delta)
    ref = optimal_assignment(Xi, Yi, .5)
    lp = gap_curve(Xp, Yp, US)
    li = gap_curve(Xi, Yi, US, reference=ref)
    ax.plot(US, lp, label="True-plan paired")
    ax.plot(US, li, label="Independent + empirical OT", ls="--")
    assert abs(lp[25]) < 1e-12 and abs(li[25]) < 1e-12
    ax.set(title=f"δ = {delta:g}", xlabel="u", ylabel="Mean gap")
axes[0].legend(fontsize=7); fig.tight_layout(); plt.show()

At $\delta=0$, the paired loss is identically zero on the PSD slice for every $n$. The independent-marginal loss need not be zero: it describes the sensitivity of a finite empirical assignment between two different clouds. Apparent curvature in the latter is not evidence of population identifiability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
delta, seed = .1, 12
# Use nested draws here to isolate point count. This differs from the article,
# which deliberately preserves the original seed=n and replicate protocol.
rng = np.random.default_rng(seed)
Xmax, Zmax = rng.standard_normal((160, 2)), rng.standard_normal((160, 2))
for n in [30, 80, 160]:
    X = Xmax[:n]
    paired = gap_curve(X, cosine_map(X, delta), US)
    Y = cosine_map(Zmax[:n], delta)
    independent = gap_curve(X, Y, US, reference=optimal_assignment(X, Y, .5))
    axes[0].plot(US, paired/delta**2, label=f"n={n}")
    axes[1].plot(US, independent/delta**2, label=f"n={n}")
axes[0].set(title="Paired: Lₙ / δ²", xlabel="u", ylabel="Normalized mean gap")
axes[1].set(title="Independent + OT: Lₙ / δ²", xlabel="u")
for ax in axes: ax.legend()
fig.tight_layout(); plt.show()

## Convexity and a sampled zero-interval diagnostic
The assignment maximum is a maximum of affine functions of $u$. Subtracting the fixed observed score leaves a convex function. On an evenly spaced grid its successive slopes must be nondecreasing, up to floating-point error.

Below we report only the extent of near-zero **evaluated grid nodes** around $1/2$. This is not an exact plateau endpoint computation. A smaller tolerance or finer grid can change this diagnostic; no monotonicity in $n$ is asserted.

In [ ]:
def zero_grid_interval(us, losses, tolerance=1e-12):
    k = np.argmin(abs(us-.5)); left = right = k
    while left > 0 and losses[left-1] <= tolerance: left -= 1
    while right+1 < len(us) and losses[right+1] <= tolerance: right += 1
    return us[left], us[right]

for n in [30, 80, 160]:
    X, Y = sample_pairs("cosine", n, seed=n, delta=.1)
    losses = gap_curve(X, Y, US)
    slopes = np.diff(losses)/np.diff(US)
    assert np.diff(slopes).min() > -1e-10
    print(f"n={n}: near-zero grid interval {zero_grid_interval(US, losses)}")

## Why a large-$n$ paired curve can still look flat
Positive $\delta$ breaks the population degeneracy, but weak perturbations produce a small signal. Finite assignments also retain a plateau around the reference. The paired and independently matched schemes remove different reference scores, so they need not resemble one another at a visually large but finite $n$.

Use the article's full $n=1000,2000$ data in notebook 03 rather than extrapolating from this quick exploration. Rescaling by $\delta^{-2}$ helps read the vertical scale; it does not change the zero set or establish a convergence rate.